# 03 · Local objects: does PointLLM read the shape, or just guess?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Md-Mosabbir/localpoint3d/blob/main/notebooks/03_pointllm_mcq_near_ood.ipynb)

**Question:** when PointLLM sees a Bangladeshi object (bodna, piri, …), does it recognise it **from its shape**, or does it answer from habits it learned in training?

Our local objects are **near-OOD** (near *out-of-distribution*). PointLLM never saw the words "bodna", "lota" or "piri" in its 660k training captions, but it saw thousands of look-alikes: teapots, vases, stools (journal 05). So the honest test isn't "does it say *bodna*", but "**can it tell a bodna from a teapot when it only has the shape?**"

**What happens in this notebook**
0. **Settings** (the next cell). This is the only cell you normally change.
1. Set up: the same PointLLM environment as notebook 01
2. Load the scans: local objects + our 17 everyday scans (the "known" reference) + the answer options
3. **OOD check:**
   - (a) are the object names in PointLLM's training captions?
   - (b) how far are our local scans from the everyday ones, in PointLLM's own "fingerprint" space?
4. **Multiple choice, grey only:** PointLLM rates 4 answers per scan (the correct one, a look-alike, a same-family object, an unrelated one)
5. **Blind test:** the same 4 answers for a featureless ball, random noise and a different object. If it picks "teapot" for a ball as well, "teapot" was never about the shape.

**How it picks without writing:** we don't let PointLLM write an answer or a letter. We measure **how likely** it finds each answer ("A teapot.", "A vase.", …) after seeing the scan. Why:
- ~10× faster than generating text
- no empty answers
- no letter bias (LLMs prefer some letters whatever the options say)

The method comes from the research in journal 05.

**Before you start:** *Runtime → Change runtime type → T4 GPU*, check the Settings cell, then *Runtime → Run all*. First run: ~15–20 min of installs + model download, then the tests take ~10–20 min.

---
# ⚙️ Settings: change things here

**Which scans**
- `LOCAL_DIR`: the Drive folder with the cleaned local scans, named like `Bodna_1_cleaned.ply`, `Piri_2_cleaned.ply`. The object name (the part before the number) must match a row in the options file.
- `EVERYDAY_DIR`: our 17 cleaned everyday scans. They are the **in-distribution** reference: same phone, same cleaning, objects PointLLM knows well. `""` = skip them.
- `ONLY_SCANS`: run only some local files. `[]` = all. `["Bodna"]` = only files whose name contains "Bodna".
- `COLOUR`:
  - `"grey"` (0.5): what we used so far.
  - `"black"` (0): what the PointLLM paper used for objects without colour.
  - `"original"`: real colours, for a comparison run.

**Multiple choice**
- `OPTIONS_FILE`: one row per object: `object, correct, look_alike, same_family, unrelated`. It's `data/mcq_options.csv` in the repo. To use your own, upload a copy to Drive and put its full path here.
- `SEEDS`: PointLLM is random inside. We use 5 seeds, not 3, because grey answers were less stable in notebook 01, and take the majority.
- `BLIND_INPUTS`: inputs with the wrong content, to catch answers that don't depend on the shape:
  - `"sphere"`: a smooth ball.
  - `"noise"`: random points.
  - `"other"`: a scan of a different object.
- `SCORE_BY`: how answers are compared.
  - `"log_prob"`: the plain likelihood of the whole answer, i.e. what PointLLM really prefers. Default.
  - `"log_prob_per_char"`: divided by the length. **Avoid**: on 31 Oct it picked the *longest* answer 71 % of the time, even for random noise. Keep the options about the same length instead (1–2 words each); the notebook warns if they aren't.
- `CHECK_EVERYDAY`: also run the test on the everyday scans. It should be easy for them. If it isn't, the test is broken, not the hypothesis.
- `CIRCULAR_EVAL`: optional extra (slow). PointLLM *writes* an answer letter, 4 times with the options rotated. A scan counts as correct only if all 4 are right.

**For the report and slides**
- Every chart and every scan picture is saved as a PNG in `RESULTS_DIR/figures_<RUN_NAME>_<date>/`, ready for slides.
- `SAVE_TO_SHEETS = True` also puts all the result tables into a new **Google Sheet** in your Drive. Google asks for permission once.

**OOD check**
- `CAPTION_WORDS`: words to count in PointLLM's 660k training captions (a 227 MB download).
- `K_NEIGHBOURS`: how many nearest everyday scans the distance is measured to.

> 🧠 A **seed** fixes the "random" choices a program makes, so the same seed always gives the same result. Different seeds = several independent tries.

In [ ]:
# ---- Which scans
USE_DRIVE = True                                               # False = upload local scans by hand (no everyday reference)
LOCAL_DIR = "/content/drive/MyDrive/Models/cleaned_bd/near_ood_set"                   # Bodna_1_cleaned.ply, Piri_1_cleaned.ply, ...
EVERYDAY_DIR = "/content/drive/MyDrive/Models/cleared_test_data"                      # our 17 chairs/cups/tables/keyboards; "" = skip
RESULTS_DIR = "/content/drive/MyDrive/localpoint3d/results"   # where the CSVs are saved
RUN_NAME = "near_ood"                                          # goes into the CSV names
ONLY_SCANS = []                                                # [] = all local scans; ["Bodna"] = only matching file names
SPECKS = "removed"                                             # drop stray floating specks before scaling
COLOUR = "grey"                                                # "grey" (0.5), "black" (0) or "original"

# ---- Multiple choice
OPTIONS_FILE = "data/mcq_options.csv"     # path inside the repo, or a full Drive path to your own copy
QUESTION = "What is this?"                # one of the questions PointLLM was trained on
SEEDS = [0, 1, 2, 3, 4]
BLIND_INPUTS = ["sphere", "noise", "other"]
SCORE_BY = "log_prob"                     # plain likelihood (fair when options have similar length)
CHECK_EVERYDAY = True                     # also test the everyday scans (should be easy)
CIRCULAR_EVAL = False                     # extra: let it write a letter, 4 rotations of the options (slow)

# ---- Report
SAVE_TO_SHEETS = False                    # True = also put the result tables into a new Google Sheet

# ---- OOD check
SEARCH_CAPTIONS = True                    # count words in PointLLM's training captions (227 MB download)
CAPTION_WORDS = ["bodna", "piri", "kula", "muura", "patil", "rehal", "hurricane lamp", "lota", "ewer",
                 "teapot", "vase", "stool", "bench", "tray", "drum", "pot", "bucket", "easel", "lantern",
                 "cup", "chair"]
K_NEIGHBOURS = 3

---
# Part 1 · Setup

## Check the GPU

`nvidia-smi` lists the graphics card. We need roughly 14 GB of GPU memory for the 7B model in float16; a T4 has 15 GB. `free -h` shows normal RAM.

> 🧠 Lines starting with `!` are not Python. They are run as shell (terminal) commands.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
!free -h | head -2

## Get our project code

Colab only opened this *notebook*. Our helper code lives in the repo's `src/` folder, so we clone the repo into the Colab machine (or update it if it's already there).

> 🧠 **Python note:** `sys.path` is the list of folders Python searches when you write `import something`. Adding `src/` to it is what lets us `import pc_io` below.
>
> 🧠 After a `!` command, `_exit_code` holds its exit code: 0 means success, anything else means it failed. We stop here if it failed, so the notebook never runs with an old copy of the code.

**The repo is private**, so Colab needs a GitHub **token** (a password just for this) to download it. Do this once per person:
1. GitHub → your picture → *Settings* → *Developer settings* → *Personal access tokens* → *Fine-grained tokens* → *Generate new token*.
2. *Repository access*: *Only select repositories* → `localpoint3d`. *Permissions* → *Contents*: **Read-only**. Generate, then copy the token.
3. In Colab: the 🔑 key icon on the left → *Add new secret* → Name `GITHUB_TOKEN`, Value = the token → switch on *Notebook access*.

Colab secrets belong to your Google account, so every notebook can use them, and they are never saved inside the notebook. Teammates need to be collaborators on the repo for their own token to work.


In [ ]:
import base64, os, sys

REPO_DIR = "/content/localpoint3d"

# The repo is private, so git needs a GitHub token: 🔑 Secrets panel -> GITHUB_TOKEN (see the text above).
# The token goes in a request header, not in the URL, so it never shows up in any output.
auth = ""
try:
    from google.colab import userdata
    basic = base64.b64encode(f"x-access-token:{userdata.get('GITHUB_TOKEN')}".encode()).decode()
    auth = f'-c http.extraHeader="Authorization: Basic {basic}"'
    print("Using the GITHUB_TOKEN secret")
except Exception as e:
    print("No GITHUB_TOKEN secret found:", type(e).__name__, "-> only works if the repo is public")

if os.path.isdir(REPO_DIR):
    !git {auth} -C {REPO_DIR} pull --ff-only
else:
    !git {auth} clone --depth 1 https://github.com/Md-Mosabbir/localpoint3d.git {REPO_DIR}
if _exit_code != 0:  # the exit code of the git command above: 0 = it worked
    raise RuntimeError("Downloading our code from GitHub failed (read the git message above). "
                       "Private repo: add the GITHUB_TOKEN secret (see the text above), then run this cell again.")

sys.path.insert(0, f"{REPO_DIR}/src")
print("Code is in", REPO_DIR)

## Get PointLLM

This is the official PointLLM code. We use HTTPS because the `git@…` SSH link in their README doesn't work on Colab.

In [ ]:
POINTLLM_DIR = "/content/PointLLM"
if not os.path.isdir(POINTLLM_DIR):
    !git clone --depth 1 https://github.com/InternRobotics/PointLLM.git {POINTLLM_DIR}
else:
    print("PointLLM already cloned")

## Build a separate Python 3.10 environment

PointLLM was written for **Python 3.10 + PyTorch 2.0.1 + an old Transformers**. Colab's own Python is newer and those versions won't install into it. So we make a separate, isolated Python 3.10 environment (a *virtual environment*, "venv") at `/content/venv` using `uv`, a fast installer.

The notebook itself keeps running on Colab's Python. Later we start the model **inside the venv** as a separate background process.

> 🧠 `%%bash` at the top turns the whole cell into a shell script. `set -e` = stop at the first error. `if [ -d … ]` checks whether the folder already exists, so running the cell twice in one session doesn't fail.

In [ ]:
%%bash
set -e
pip install uv
if [ -d /content/venv ]; then
    echo "/content/venv already exists (this session ran the cell before), keeping it"
else
    uv venv --python 3.10 /content/venv
fi


## Install PointLLM's dependencies into that environment (~5 min)

- `torch==2.0.1` matches what PointLLM was tested with
- `-e .` installs the PointLLM repo itself plus everything in its requirements
- `plyfile`, `trimesh` are for reading our scan files (used by `src/pc_io.py`)
- `numpy<2` everywhere because the old libraries break with NumPy 2

In [ ]:
%%bash
set -e
cd /content/PointLLM
PY=/content/venv/bin/python
uv pip install --python $PY "torch==2.0.1" "numpy<2"
uv pip install --python $PY -e . "numpy<2"
uv pip install --python $PY accelerate plyfile trimesh matplotlib "numpy<2"

## Smoke test

Quick check that everything imports **before** we spend 10 minutes downloading model weights. You should see `imports OK`. Warnings about deprecated functions are fine.

In [ ]:
%%bash
cd /content/PointLLM
/content/venv/bin/python -c "
import torch, transformers, pointllm, plyfile, trimesh
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__)
print('imports OK')"

---
# Part 2 · Our scans

## Install the file readers in Colab's own Python

The venv has them already. The notebook's Python needs them too, so we can preview scans here.

In [ ]:
!pip install plyfile trimesh

## Connect Google Drive

Our scans live on Google Drive. *Mounting* Drive makes it appear inside Colab as a normal folder, `/content/drive/MyDrive`. A Google popup asks for permission: allow it.

No Drive? Set `USE_DRIVE = False` in the Settings cell and a file picker opens to upload scans from your computer instead (they disappear when the session ends).

In [ ]:
if USE_DRIVE:  # set in the Settings cell
    from google.colab import drive
    drive.mount("/content/drive")
else:
    from google.colab import files
    os.makedirs("/content/scans", exist_ok=True)
    for name, data in files.upload().items():  # opens a file picker
        with open(os.path.join("/content/scans", name), "wb") as fh:
            fh.write(data)

## Find the scans

Lists the local scans (`LOCAL_DIR`) and the everyday scans (`EVERYDAY_DIR`), including subfolders, and shows which **object** each file name is read as. The object name decides which answer options a scan gets.

It also makes the folder where this run's charts and pictures are saved (`FIG_DIR`).

Teammates: if the folders are shared from Md's Drive, add a shortcut to *My Drive* first (drive.google.com → right-click the folder → *Organize* → *Add shortcut*), so the same paths work for everyone.

In [ ]:
import glob
from datetime import datetime
import charts
from pc_io import parse_scan_name

if not USE_DRIVE:
    LOCAL_DIR, EVERYDAY_DIR, RESULTS_DIR = "/content/scans", "", "/content/results"
RUN_STAMP = f"{RUN_NAME}_{datetime.now():%Y-%m-%d_%H%M}"
FIG_DIR = f"{RESULTS_DIR}/figures_{RUN_STAMP}"  # every chart and picture of this run goes here
os.makedirs(FIG_DIR, exist_ok=True)


def save_figure(figure, name):
    """Save a chart (matplotlib) or a picture (PIL image) as FIG_DIR/<name>.png for slides."""
    path = f"{FIG_DIR}/{name}.png"
    if hasattr(figure, "savefig"):
        charts.save(figure, path)
    else:
        figure.save(path)
    print("saved", path)
EXTS = (".ply", ".obj", ".glb", ".gltf", ".off", ".stl")


def find_scans(folder):
    if not folder:
        return []
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"No folder at {folder}. Fix the path in the Settings cell (copy it from the Files panel).")
    return sorted(p for p in glob.glob(f"{folder}/**/*", recursive=True) if p.lower().endswith(EXTS))


def object_of(path):
    return parse_scan_name(path)["object"]


LOCAL_SCANS = find_scans(LOCAL_DIR)
if ONLY_SCANS:
    LOCAL_SCANS = [p for p in LOCAL_SCANS if any(word in os.path.basename(p) for word in ONLY_SCANS)]
EVERYDAY_SCANS = find_scans(EVERYDAY_DIR)

for name, scans in [("Local", LOCAL_SCANS), ("Everyday", EVERYDAY_SCANS)]:
    print(f"{name}: {len(scans)} scan(s)")
    for p in scans:
        print(f"   {os.path.basename(p):32s} -> object '{object_of(p)}'")

## Load the answer options

Every object gets the same 4 kinds of answer:

| Column | Meaning | Bodna example |
|---|---|---|
| `correct` | the right answer **in plain English**. PointLLM can't know the word "bodna", so asking for it would test vocabulary, not shape | spouted water pot |
| `look_alike` | a known object with a similar shape: the hard trap | teapot |
| `same_family` | similar use or category | vase |
| `unrelated` | clearly different. If it picks this, something is broken | chair |

Keep all 4 options about the **same length** (1–2 words). A long answer has more words that can each be "unlikely", so it loses on likelihood even when it is right. With 4 options, guessing gets **25 %** right. A missing row means that object is skipped. To change options, edit `data/mcq_options.csv` on GitHub (then re-run the "Get our project code" cell), or point `OPTIONS_FILE` at your own copy.

> 🧠 `pd.read_csv` reads a table into a pandas **DataFrame** (rows and named columns, like a spreadsheet). `.set_index("object")` lets us look rows up by name: `OPTIONS.loc["bodna", "correct"]`.

In [ ]:
import pandas as pd
from IPython.display import display

options_path = OPTIONS_FILE if os.path.isabs(OPTIONS_FILE) else f"{REPO_DIR}/{OPTIONS_FILE}"
OPTIONS = pd.read_csv(options_path).set_index("object")
ROLES = [c for c in OPTIONS.columns if c != "note"]  # correct, look_alike, same_family, unrelated
assert ROLES[0] == "correct", "The first answer column of the options file must be 'correct'"
CHANCE = 1 / len(ROLES)
display(OPTIONS[ROLES])
print(f"{len(ROLES)} options per object -> guessing gets {CHANCE:.0%} right")

import mcq_analysis
for obj, words in mcq_analysis.length_warnings(OPTIONS, ROLES):
    print(f"⚠️ The options for '{obj}' differ a lot in length {words}. Long answers get lower likelihoods, so "
          "the comparison is unfair. Make them 1-2 words each.")

missing = sorted({object_of(p) for p in LOCAL_SCANS + EVERYDAY_SCANS} - set(OPTIONS.index))
if missing:
    print("⚠️ No options for", missing, "-> those scans are skipped in the multiple choice. Add rows to", OPTIONS_FILE)


def as_answer(option):
    """'teapot' -> 'A teapot.'  PointLLM's short training captions look like that ("A green and white rifle.")."""
    article = "An" if option[0].lower() in "aeiou" else "A"
    return f"{article} {option}."


print("Answers are scored as, e.g.:", [as_answer(o) for o in OPTIONS.iloc[0][ROLES]])

## See what the model will see

One picture per local object, in `COLOUR`, plus the two blind inputs. Each picture is also saved to `FIG_DIR` for the slides. Check that every object is upright and that no floor or wall is left. The blind inputs should look like a ball and a cloud of dots.

In [ ]:
from pointllm_helpers import preprocess, preview, fake_cloud

shown = set()
for p in LOCAL_SCANS:
    if object_of(p) not in shown:
        shown.add(object_of(p))
        pts, _ = preprocess(p, colour=COLOUR, specks=SPECKS)
        picture = preview(pts, title=f"{os.path.basename(p)} · {COLOUR}")
        display(picture)
        save_figure(picture, f"scan_{object_of(p)}")
for kind in ["sphere", "noise"]:
    picture = preview(fake_cloud(kind, colour=COLOUR), title=f"blind input: {kind}")
    display(picture)
    save_figure(picture, f"blind_{kind}")

---
# Part 3 · Are our scans out-of-distribution?

## (a) Did PointLLM ever read these words?

PointLLM learned to name objects from ~660k short captions ("A green and white rifle."). We download those exact captions and count how many mention each word in `CAPTION_WORDS`.
- **0** = the model never saw the word, so it can't name the object that way.
- **Thousands** = a well-known object.

Near-OOD objects have a name count of 0 but common look-alikes. This runs in Colab's own Python and doesn't need the model.

> 🧠 `re.search(r"\bstool\b", text)` looks for the *whole word* "stool": `\b` means "word boundary", so "stools" and "toadstool" don't count.

In [ ]:
import json, re
import matplotlib.pyplot as plt

if SEARCH_CAPTIONS:
    from huggingface_hub import hf_hub_download
    caption_file = hf_hub_download("RunsenXu/PointLLM", "PointLLM_brief_description_660K_filtered.json",
                                   repo_type="dataset")
    with open(caption_file) as fh:
        training = json.load(fh)
    captions = [turn["value"].lower() for item in training for turn in item["conversations"] if turn["from"] == "gpt"]
    word_counts = pd.DataFrame(
        [{"word": w, "captions": sum(1 for c in captions if re.search(rf"\b{re.escape(w.lower())}\b", c))}
         for w in CAPTION_WORDS])
    print(f"{len(captions):,} training captions")
    display(word_counts)
    fig = charts.caption_counts(word_counts)
    save_figure(fig, "1_training_words")
    plt.show()
else:
    print("Skipped (SEARCH_CAPTIONS = False)")

## Start the model in the background

Same as notebook 01: `src/pointllm_worker.py` runs in the Python 3.10 environment and keeps the model loaded. This notebook asks it for three things:
- `embed_pointllm(path)`: a **fingerprint** of the scan. PointLLM's point encoder turns the cloud into 513 tokens of 384 numbers, and we average them into one vector.
- `score_pointllm(path, answers)`: how likely PointLLM finds each answer after seeing the scan (a **log-probability**: 0 = certain, more negative = less likely).
- `ask_pointllm(path, prompt)`: a normal written answer (only for the optional CircularEval).

All three use `COLOUR` and `SPECKS` from Settings. A path like `"fake:sphere"` means a blind input instead of a file.

> 🧠 A **log-probability** is the logarithm of a probability. Probabilities of long answers get tiny (0.001 × 0.02 × …), and logs turn those multiplications into sums, which computers handle better. −1 ≈ 37 %, −3 ≈ 5 %, −7 ≈ 0.1 %.

In [ ]:
import subprocess
import numpy as np


def start_worker():
    env = {**os.environ, "POINTLLM_DIR": POINTLLM_DIR}
    w = subprocess.Popen(["/content/venv/bin/python", "-u", f"{REPO_DIR}/src/pointllm_worker.py"],
                         cwd=POINTLLM_DIR, env=env, text=True, bufsize=1,
                         stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT)
    print("Loading the model (first time: downloads ~14 GB, be patient)...")
    for line in w.stdout:
        if "@@READY@@" in line:
            print("✅ Model ready.")
            return w
        if line.strip():
            print("  ", line.rstrip()[:200])
    raise RuntimeError("The worker stopped while loading. Read the log above.")


def send(request):
    """Send one request to the worker and wait for its result."""
    request = {"colour": COLOUR, "specks": SPECKS, "rotation": "None", "floor": "kept", **request}
    worker.stdin.write(json.dumps(request) + "\n")
    worker.stdin.flush()
    for line in worker.stdout:
        if line.startswith("@@RESULT@@"):
            res = json.loads(line[len("@@RESULT@@"):])
            if "error" in res:
                raise RuntimeError(res["error"])
            return res
    raise RuntimeError("The worker died. Re-run this cell to restart it.")


def embed_pointllm(path, seed=0):
    return np.array(send({"task": "embed", "path": path, "prompt": "", "seed": seed})["vector"])


def score_pointllm(path, answers, seed=0):
    return send({"task": "score", "path": path, "prompt": QUESTION, "answers": answers, "seed": seed})["scores"]


def ask_pointllm(path, prompt, seed=0):
    return send({"task": "ask", "path": path, "prompt": prompt, "seed": seed})["answer"]


if "worker" not in globals() or worker.poll() is not None:  # don't load twice
    worker = start_worker()

## (b) Fingerprints: how far is each local scan from the known objects?

For every scan we take PointLLM's fingerprint, averaged over `SEEDS` because the encoder is random. Then we measure the **distance to the `K_NEIGHBOURS` nearest everyday scans**:
- small = "looks like something it knows"
- large = "unlike anything in the reference set"

**What counts as far?** We first measure the same distance for each everyday scan to the *other* everyday scans (leaving itself out). A local scan farther away than 95 % of those is marked **OOD**.

This is the kNN method from Sun et al. 2022, *Out-of-distribution detection with deep nearest neighbors*. The reference here is our 17 everyday scans, not all of PointLLM's training data, so it measures "new shape compared to scans we know it handles".

> 🧠 **Cosine distance** = 1 − cos(angle between two vectors). It's 0 when they point the same way, about 1 when unrelated. We subtract the average everyday fingerprint first, so the shared part that every scan has doesn't hide the differences.

In [ ]:
everyday = list(EVERYDAY_SCANS)
all_scans = everyday + LOCAL_SCANS
raw = {}
for p in all_scans:
    raw[p] = np.mean([embed_pointllm(p, seed) for seed in SEEDS], axis=0)
    print("fingerprint:", os.path.basename(p))

centre = np.mean([raw[p] for p in (everyday or all_scans)], axis=0)
FINGERPRINT = {p: (raw[p] - centre) / np.linalg.norm(raw[p] - centre) for p in all_scans}


def distances(p, refs):
    """Cosine distance from scan p to every scan in refs, smallest first: [(distance, ref), ...]."""
    return sorted((1 - float(FINGERPRINT[p] @ FINGERPRINT[r]), r) for r in refs if r != p)


rows = []
for p in all_scans:
    near = distances(p, everyday)
    if not near:
        continue
    k = min(K_NEIGHBOURS, len(near))
    rows.append({"set": "everyday" if p in everyday else "local", "file": os.path.basename(p), "object": object_of(p),
                 "knn_distance": np.mean([d for d, _ in near[:k]]),
                 "nearest_everyday": os.path.basename(near[0][1]), "nearest_distance": near[0][0]})
ood = pd.DataFrame(rows)
if len(ood) and (ood.set == "everyday").sum() >= 2:
    threshold = np.percentile(ood[ood.set == "everyday"].knn_distance, 95)
    ood["OOD"] = ood.knn_distance > threshold
    print(f"OOD if the distance is above {threshold:.3f} (95 % of everyday scans are closer)")
    display(ood.sort_values(["set", "knn_distance"]))
    fig = charts.ood_distances(ood, threshold)
    save_figure(fig, "2_ood_distance")
    plt.show()
else:
    print("Need at least 2 everyday scans as a reference. Set EVERYDAY_DIR in Settings.")

## Map of all fingerprints

384 numbers per scan can't be drawn. **PCA** squeezes them into the 2 directions where the scans differ most, so we can plot them. Scans of the same object should form a group.
- A local object sitting **inside** a known group (e.g. bodna among the cups) is likely to be **mistaken** for that object.
- A local object **on its own** is something new.

> 🧠 `np.linalg.svd` finds those directions (the *principal components*). `X @ vt[:2].T` projects every fingerprint onto the first two.

In [ ]:
X = np.stack([FINGERPRINT[p] for p in all_scans])
X = X - X.mean(axis=0)
_, _, vt = np.linalg.svd(X, full_matrices=False)
xy = X @ vt[:2].T

fig, ax = plt.subplots(figsize=(8, 6))
for obj in sorted({object_of(p) for p in all_scans}):
    idx = [i for i, p in enumerate(all_scans) if object_of(p) == obj]
    is_local = all_scans[idx[0]] in LOCAL_SCANS
    ax.scatter(xy[idx, 0], xy[idx, 1], s=90 if is_local else 40, marker="*" if is_local else "o", label=obj)
    for i in idx:
        ax.annotate(os.path.splitext(os.path.basename(all_scans[i]))[0].replace("_cleaned", ""), xy[i],
                    fontsize=7, alpha=0.7)
ax.set_title("PointLLM fingerprints (stars = local objects)")
ax.legend(fontsize=8)
save_figure(fig, "3_fingerprint_map")
plt.show()

---
# Part 4 · Multiple choice (grey) + blind test

## Build the list of tests

For every scan whose object has a row in the options file:
- **real**: the scan itself.
- **blind** (once per object, for each of `BLIND_INPUTS`): the same 4 answers, but for a ball, noise, or a scan of a *different* object.

The blind inputs tell us what PointLLM picks **without** the right shape. Each test is scored with every seed and every answer. The cell prints how many model passes that makes.

In [ ]:
import random

sets = [("local", LOCAL_SCANS)] + ([("everyday", EVERYDAY_SCANS)] if CHECK_EVERYDAY else [])
jobs = []  # (set, object, input kind, path)
for set_name, scans in sets:
    scans = [p for p in scans if object_of(p) in OPTIONS.index]
    for p in scans:
        jobs.append((set_name, object_of(p), "real", p))
    for obj in sorted({object_of(p) for p in scans}):
        for kind in BLIND_INPUTS:
            if kind == "other":  # a scan of some other object, picked the same way every run
                others = [p for p in LOCAL_SCANS + EVERYDAY_SCANS if object_of(p) != obj]
                if others:
                    jobs.append((set_name, obj, "other", random.Random(obj).choice(others)))
            else:
                jobs.append((set_name, obj, kind, f"fake:{kind}"))

n_passes = len(jobs) * len(SEEDS) * len(ROLES)
print(f"{len(jobs)} inputs x {len(SEEDS)} seeds x {len(ROLES)} answers = {n_passes} model passes, "
      f"roughly {n_passes * 0.5 / 60:.0f}-{n_passes * 1.0 / 60:.0f} min on a T4")

## Run it → CSV

One row per input × seed × answer, with the log-probability. The CSV is saved to `RESULTS_DIR` as `pointllm_mcq_<RUN_NAME>_<date>.csv`. Download it to `results/` in the repo when you're done.

> 🧠 `tqdm` wraps a loop and draws a progress bar. `zip(a, b)` walks through two lists side by side.

In [ ]:
from tqdm.auto import tqdm

rows = []
for set_name, obj, kind, path in tqdm(jobs):
    options = [OPTIONS.loc[obj, role] for role in ROLES]
    answers = [as_answer(o) for o in options]
    for seed in SEEDS:
        for role, option, answer, (log_prob, n_tokens) in zip(ROLES, options, answers,
                                                               score_pointllm(path, answers, seed)):
            rows.append({"set": set_name, "object": obj, "input": kind,
                         "file": path if kind in ("sphere", "noise") else os.path.basename(path),
                         "colour": COLOUR, "question": QUESTION, "seed": seed, "role": role,
                         "option": option, "answer": answer, "log_prob": log_prob, "n_tokens": n_tokens})

mcq = pd.DataFrame(rows)
mcq["log_prob_per_char"] = mcq.log_prob / mcq.answer.str.len()
csv_path = f"{RESULTS_DIR}/pointllm_mcq_{RUN_STAMP}.csv"
mcq.to_csv(csv_path, index=False)
print("Saved", len(mcq), "rows to", csv_path)

## Results 1: what did it pick?

For each input and seed, the answer with the best `SCORE_BY` is PointLLM's **pick**. The **majority over seeds** is the final pick. `seeds_agreeing` says how many seeds chose it: 5/5 is a firm answer, 2/5 is a coin toss.

In the table, `pick` is for the real scan. The `sphere`, `noise` and `other` columns are what PointLLM picked for the blind inputs **of the same object**.
- `same_as_sphere = True` is a warning sign: it would have said the same about a ball.

The calculations are in `src/mcq_analysis.py`, so the CLIP notebook and re-analyses of saved CSVs use exactly the same code.

> 🧠 Inside, `groupby([...])` splits the table into groups (one per input and seed), and `idxmax()` finds the row with the largest score in each group.

In [ ]:
best = mcq_analysis.picks(mcq, SCORE_BY)       # every input x seed: the answer with the best score
vote = mcq_analysis.vote(best, OPTIONS)        # majority over seeds
table = mcq_analysis.per_scan(vote)            # one row per real scan + what the blind inputs got
display(table)

## Results 2: better than guessing?

**Binomial test:** if PointLLM were only guessing, each scan would be right with probability 25 %. The p-value is the chance of getting at least this many right by pure guessing.
- p < 0.05 is the usual line for "not just luck".
- With 10 scans you need at least 6 right; with 5 scans, at least 4.

Scans of the same object aren't fully independent (take 2 of a bodna is a lot like take 1). So we also count per **object** (majority of its takes), which is the stricter number.

We print the blind inputs' accuracy next to it: a ball should be at about 25 % or below. The chart shows the same numbers, which makes it the main slide.

> 🧠 Inside `mcq_analysis`, `binomtest(k, n, p)` from **scipy** computes exactly that probability. `alternative="greater"` means we only care whether it is *better* than guessing.

In [ ]:
summary = mcq_analysis.accuracy_summary(table, vote, CHANCE, BLIND_INPUTS)
for set_name, s in summary.groupby("set", sort=False):
    print(f"{set_name.upper()} (guessing = {CHANCE:.0%})")
    for r in s.itertuples():
        print(f"  {r.label:28s} {r.correct:3d}/{r.n:<3d} = {r.accuracy:4.0%}   p = {r.p_value:.3f}"
              + ("   ✅ better than guessing" if r.p_value < 0.05 else ""))

fig = charts.accuracy_by_input(summary, CHANCE)
save_figure(fig, "4_accuracy_real_vs_blind")
plt.show()

## Results 3: calibrated: how much did the real shape help each answer?

Every answer has some likelihood even with no object at all, because some phrases are simply more common. "A chair." is a very common caption, "A cane stool." is not. To remove that, we subtract what PointLLM says about "nothing", the average of the ball and the noise:

**calibrated = log-prob(answer | real scan) − average log-prob(answer | ball, noise)**

The same answer text is on both sides, so how common the phrase is cancels out.
- If `correct` has the highest calibrated score, the shape pushed PointLLM toward the right answer, even if a common word still won the plain vote.
- `margin` = calibrated of correct − the best calibrated wrong answer. Positive = the shape favours the right answer.

**Sign test:** if the shape didn't matter, the margin would be positive about half the time.

This is a second view, not a replacement. On 31 Oct it was slightly harsher than the plain likelihood on the everyday scans (11 vs 14 of 17).

In [ ]:
if {"sphere", "noise"} <= set(BLIND_INPUTS):
    per_scan = mcq_analysis.calibrated_margins(mcq, ROLES)
    display(per_scan.round(2))
    for set_name, t in per_scan.groupby(level="set"):
        k, n, p = mcq_analysis.sign_test(t.margin)
        print(f"{set_name}: the shape favoured the correct answer for {k}/{n} scans (sign test p = {p:.3f})")
    fig = charts.shape_margin(per_scan)
    save_figure(fig, "5_shape_boost")
    plt.show()
else:
    print('Needs "sphere" and "noise" in BLIND_INPUTS.')

## Results 4: which wrong answer does it fall for?

Rows = the real object, columns = what PointLLM picked (counted over scans).
- Mostly `look_alike` = it sees the shape roughly right but files it under the familiar look-alike: the classic near-OOD failure.
- Mostly `unrelated` = it isn't reading the shape at all.

The chart counts every seed separately: 2 bodna scans × 5 seeds = 10 picks.

In [ ]:
for set_name in table.set.unique():
    t = table[table.set == set_name]
    print(set_name)
    display(pd.crosstab(t.object, pd.Categorical(t.pick, categories=ROLES)).rename_axis(columns="picked"))
    seed_picks = best[(best.set == set_name) & (best.input == "real")]
    fig = charts.picks_by_object(seed_picks, ROLES, title=f"What does PointLLM pick? ({set_name} objects)")
    save_figure(fig, f"6_picks_{set_name}")
    plt.show()

## (Optional) CircularEval: let it write the letter

Only runs if `CIRCULAR_EVAL = True`. Here PointLLM sees the options as a list (A–D) and **writes** its answer. We ask 4 times, rotating the list so every option is "A" once. A scan counts as correct only if all 4 rotations (majority over seeds) pick the right option. This is MMBench's *CircularEval*, which cancels out any favourite letter.

Expect it to be harsher than the likelihood test. If the two disagree a lot, the letters are the problem, not the shape. It's slow: `scans × 4 rotations × seeds` written answers at ~3 s each.

In [ ]:
LETTERS = "ABCDEFGH"

def circular_prompt(options):
    lines = [f"{LETTERS[i]}. {o}" for i, o in enumerate(options)]
    return QUESTION + "\nChoose one option:\n" + "\n".join(lines) + "\nAnswer with the letter only."


def read_choice(answer, options):
    """Which option did a written answer choose? Its index, or None if we can't tell."""
    named = [i for i, o in enumerate(options) if o.lower() in answer.lower()]
    if len(named) == 1:
        return named[0]
    m = re.match(r"\s*\(?([A-H])\)?(?:[.):]|\s*$)", answer)  # "B", "B.", "(B)", "B)"
    if m and LETTERS.index(m.group(1)) < len(options):
        return LETTERS.index(m.group(1))
    return None


if CIRCULAR_EVAL:
    circ = []
    for set_name, obj, kind, path in tqdm([j for j in jobs if j[2] == "real"]):
        options = [OPTIONS.loc[obj, role] for role in ROLES]
        for shift in range(len(ROLES)):
            order = ROLES[shift:] + ROLES[:shift]
            shown = [OPTIONS.loc[obj, r] for r in order]
            for seed in SEEDS:
                answer = ask_pointllm(path, circular_prompt(shown), seed)
                choice = read_choice(answer, shown)
                circ.append({"set": set_name, "object": obj, "file": os.path.basename(path), "shift": shift,
                             "seed": seed, "answer": answer, "pick": order[choice] if choice is not None else "unreadable"})
    circ = pd.DataFrame(circ)
    circ.to_csv(csv_path.replace(".csv", "_circular.csv"), index=False)
    per_shift = circ.groupby(["set", "object", "file", "shift"]).pick.agg(lambda r: r.value_counts().idxmax())
    circular = (per_shift == "correct").groupby(["set", "object", "file"]).all().rename("circular_correct")
    display(circular.reset_index())
    print("Unreadable answers:", (circ.pick == "unreadable").sum(), "of", len(circ))
    from scipy.stats import binomtest
    for set_name, c in circular.groupby(level="set"):
        k, n = int(c.sum()), len(c)
        print(f"{set_name}: circular {k}/{n} correct, p = {binomtest(k, n, CHANCE, alternative='greater').pvalue:.3f}")
else:
    print("Skipped (CIRCULAR_EVAL = False)")

---
# Part 5 · Save everything for the report

The tables go to `RESULTS_DIR` as CSVs next to the raw scores. Upload any CSV to **Google Sheets** with *File → Import*, or set `SAVE_TO_SHEETS = True` to get a ready Google Sheet with one tab per table.

| File / tab | What's in it |
|---|---|
| `summary` | % correct for real scans and blind inputs, with p-values: the headline numbers |
| `per_scan` | every scan's pick, how many seeds agreed, and what the blind inputs got |
| `shape_boost` | the shape boost per scan and answer |
| `ood` | fingerprint distances and the OOD flag |
| `training_words` | caption counts |
| `all_scores` | every log-probability (the raw data) |

The PNGs in `FIG_DIR` are numbered in the order they make sense on slides.

> 🧠 `gspread` is a library that edits Google Sheets from Python. `auth.authenticate_user()` asks you (once) to let this Colab session use your Google account.

In [ ]:
tables = {"summary": globals().get("summary"), "per_scan": table,
          "shape_boost": per_scan.reset_index() if "per_scan" in globals() else None,
          "ood": globals().get("ood"), "training_words": globals().get("word_counts"), "all_scores": mcq}
tables = {name: df for name, df in tables.items() if df is not None and len(df)}
for name, df in tables.items():
    if name != "all_scores":  # that one is already saved as csv_path
        df.to_csv(f"{RESULTS_DIR}/pointllm_mcq_{RUN_STAMP}_{name}.csv", index=False)
print("Tables saved to", RESULTS_DIR)

if SAVE_TO_SHEETS:
    from google.colab import auth
    from google.auth import default
    import gspread
    auth.authenticate_user()
    creds, _ = default()
    sheet = gspread.authorize(creds).create(f"localpoint3d · PointLLM MCQ · {RUN_STAMP}")
    for i, (name, df) in enumerate(tables.items()):
        tab = sheet.sheet1 if i == 0 else sheet.add_worksheet(title=name, rows=len(df) + 1, cols=len(df.columns))
        if i == 0:
            tab.update_title(name)
        tab.update([df.columns.tolist()] + df.round(4).astype(str).values.tolist())
    print("✅ Google Sheet:", sheet.url)

print("\nFigures for the slides:")
for f in sorted(os.listdir(FIG_DIR)):
    print("  ", f)

---
## Stop the model (frees the GPU)

In [ ]:
worker.terminate()
print("Worker stopped.")

---
## How to read the results

| What you see | What it means |
|---|---|
| Local scans correct, blind inputs at chance, shape boost favours correct | PointLLM **reads the shape** of our local objects |
| Mostly `look_alike` picks, but the shape boost still favours `correct` | It sees the shape, but the familiar word wins. This is the near-OOD effect |
| Same pick for the real scan and the ball (`same_as_sphere`) | The answer comes from language habits, not the shape |
| Everyday scans fail too | Something is wrong with the test (options, colour, preprocessing), not with the local objects |
| Many seeds disagree (`seeds_agreeing` 2–3 of 5) | Too unstable to call. Try `COLOUR = "black"` |

Write the numbers into `journal/05_near_ood_mcq.md`, and copy the CSV to `results/`.

## Troubleshooting

| Problem | Try |
|---|---|
| `No folder at …` | Fix `LOCAL_DIR` / `EVERYDAY_DIR` in Settings (copy the path from the Files panel). |
| A local object is missing from the results | Its name isn't in the options file. The name is the part of the file name before the number: `Bodna_1_cleaned.ply` → `bodna`. |
| `CUDA out of memory` | Runtime → Restart session and run again. Make sure no other worker is running. |
| Worker error about `task` or `score_answers` | The repo copy is old. Re-run "Get our project code" and restart the worker cell. |
| Everything picks the same answer, even for the ball | That answer is just a very common phrase. Look at the shape-boost table, which removes that effect. |